#ANOTACION FUNCIONAL

In [2]:
import os
import re
import csv
from Bio import Entrez, SeqIO

# Tu correo (obligatorio para acceder a NCBI)
Entrez.email = "tucorreo@ejemplo.com"  # 🔁 Reemplaza por el tuyo

# Rutas
carpeta_aln = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Alineamientos"
carpeta_salida = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion"
os.makedirs(carpeta_salida, exist_ok=True)

# CSV general para acumular todos los resultados
csv_general = []
header_csv = ["Archivo_aln", "Accession", "Organismo", "Gen", "Producto", "Proteína"]

# Función para obtener anotación desde NCBI
def obtener_anotacion_ncbi(acc):
    datos_txt = []
    datos_csv = []

    try:
        handle = Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text")
        record = SeqIO.read(handle, "genbank")
        handle.close()

        organismo = record.annotations.get('organism', 'Desconocido')
        descripcion = record.description

        datos_txt.append(f"🧬 ID: {acc}")
        datos_txt.append(f"Organismo: {organismo}")
        datos_txt.append(f"Descripción: {descripcion}")

        cds_list = []
        for feature in record.features:
            if feature.type == "CDS":
                gen = feature.qualifiers.get("gene", ["ND"])[0]
                producto = feature.qualifiers.get("product", ["Desconocido"])[0]
                prot_id = feature.qualifiers.get("protein_id", ["ND"])[0]
                cds_list.append(f"  - Gen: {gen}, Producto: {producto}, Proteína: {prot_id}")
                datos_csv.append([acc, organismo, gen, producto, prot_id])

        if cds_list:
            datos_txt.append("Funciones anotadas:\n" + "\n".join(cds_list))
        else:
            datos_txt.append("Sin CDS anotados.")
            datos_csv.append([acc, organismo, "ND", "ND", "ND"])

    except Exception as e:
        datos_txt.append(f"❌ Error al obtener {acc}: {e}")
        datos_csv.append([acc, "Error", "ND", "ND", "ND"])

    return "\n".join(datos_txt), datos_csv

# Extraer IDs de accesión desde archivo .aln
def extraer_ids_accesion(path_aln):
    accesiones = set()
    with open(path_aln, "r") as f:
        for linea in f:
            campos = linea.strip().split()
            if not campos:
                continue
            header = campos[0]
            if header == "9":
                continue  # ignorar la secuencia original
            match = re.match(r"([A-Z]{1,2}_?\d+\.\d+)", header)
            if match:
                accesiones.add(match.group(1))
    return sorted(accesiones)

# Procesar archivos .aln
for archivo in os.listdir(carpeta_aln):
    if archivo.endswith(".aln"):
        nombre_base = os.path.splitext(archivo)[0]
        ruta_aln = os.path.join(carpeta_aln, archivo)
        print(f"\n🔍 Procesando: {archivo}")

        accesiones = extraer_ids_accesion(ruta_aln)
        anotaciones_txt = []
        anotaciones_csv = []

        for acc in accesiones:
            anotacion_txt, anotacion_csv = obtener_anotacion_ncbi(acc)
            anotaciones_txt.append(anotacion_txt)
            anotaciones_csv.extend(anotacion_csv)
            # Guardar en CSV general
            for fila in anotacion_csv:
                csv_general.append([nombre_base] + fila)

        # Guardar archivo .txt individual
        ruta_txt = os.path.join(carpeta_salida, f"{nombre_base}_anotacion.txt")
        with open(ruta_txt, "w", encoding="utf-8") as f:
            f.write("\n\n".join(anotaciones_txt))

        # Guardar archivo .csv individual
        ruta_csv = os.path.join(carpeta_salida, f"{nombre_base}_anotacion.csv")
        with open(ruta_csv, "w", newline='', encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(header_csv[1:])  # sin nombre del archivo
            writer.writerows(anotaciones_csv)

        print(f"✅ Guardado: {ruta_txt} y {ruta_csv}")

# Guardar archivo CSV general con todas las anotaciones
ruta_csv_general = os.path.join(carpeta_salida, "anotacion_completa.csv")
with open(ruta_csv_general, "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header_csv)
    writer.writerows(csv_general)

print(f"\n📦 Anotación general guardada: {ruta_csv_general}")




🔍 Procesando: query_10_top5_alignment.aln
✅ Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\query_10_top5_alignment_anotacion.txt y C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\query_10_top5_alignment_anotacion.csv

🔍 Procesando: query_11_top5_alignment.aln
✅ Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\query_11_top5_alignment_anotacion.txt y C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_pato

In [3]:
import os
import csv
from collections import Counter

# Ruta al archivo anotado general
csv_anotacion = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\anotacion_completa.csv"
carpeta_salida = os.path.dirname(csv_anotacion)

# Cargar anotaciones
organismos = []
genes = []
productos = []

with open(csv_anotacion, newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        organismos.append(row["Organismo"])
        if row["Gen"] != "ND":
            genes.append(row["Gen"])
        if row["Producto"] != "Desconocido":
            productos.append(row["Producto"])

# Contar ocurrencias
conteo_organismos = Counter(organismos)
conteo_genes = Counter(genes)
conteo_productos = Counter(productos)

# Guardar resúmenes
def guardar_conteo(conteo, nombre_archivo, columna="Elemento"):
    ruta = os.path.join(carpeta_salida, nombre_archivo)
    with open(ruta, "w", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([columna, "Frecuencia"])
        for item, freq in conteo.most_common():
            writer.writerow([item, freq])
    print(f"✅ Guardado: {ruta}")

guardar_conteo(conteo_organismos, "resumen_organismos.csv", "Organismo")
guardar_conteo(conteo_genes, "resumen_genes.csv", "Gen")
guardar_conteo(conteo_productos, "resumen_productos.csv", "Producto")


✅ Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_organismos.csv
✅ Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_genes.csv
✅ Guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_productos.csv


In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Ruta base
carpeta_resumenes = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion"

# Archivos de resumen
archivos = {
    "Organismo": "resumen_organismos.csv",
    "Gen": "resumen_genes.csv",
    "Producto": "resumen_productos.csv"
}

def graficar_resumen(nombre, archivo_csv, top_n=10):
    path = os.path.join(carpeta_resumenes, archivo_csv)
    df = pd.read_csv(path)

    # Seleccionar los top N
    df_top = df.head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(df_top.iloc[::-1, 0], df_top.iloc[::-1, 1], color="#69b3a2")
    plt.xlabel("Frecuencia")
    plt.title(f"Top {top_n} {nombre}s más frecuentes")
    plt.tight_layout()

    salida_png = os.path.join(carpeta_resumenes, f"{archivo_csv.replace('.csv', '')}.png")
    plt.savefig(salida_png)
    plt.close()
    print(f"📊 Gráfico guardado: {salida_png}")

# Crear gráficos
for nombre, archivo in archivos.items():
    graficar_resumen(nombre, archivo, top_n=10)


📊 Gráfico guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_organismos.png
📊 Gráfico guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_genes.png
📊 Gráfico guardado: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_2.0\Anotacion\resumen_productos.png
